# Projeto M2 - Segmentacao e contagem de hemacias com PDI classica

**Disciplina:** Processamento de Imagens  
**Professor:** Felipe Viel  
**Autores:** preencher os nomes dos integrantes antes da entrega  
**Dataset:** Blood Cell Segmentation  

Este notebook apresenta uma pipeline final, reproduzivel e configuravel para identificar regioes correspondentes a hemacias em imagens microscopicas coloridas. A saida principal e uma mascara binaria e uma estimativa da quantidade de hemacias separadas pelo Watershed.

## 1. Enunciado e objetivo da entrega

O trabalho solicita uma pipeline completa de processamento digital de imagens reais, utilizando tecnicas classicas. Para o dataset de celulas sanguineas, a solucao deve gerar uma mascara binaria contendo as celulas de interesse e avaliar o resultado por metricas como Dice, IoU e/ou contagem de objetos.

As etapas obrigatorias contempladas aqui sao:

1. pre-processamento e conversao de espaco de cor;
2. filtragem no dominio da frequencia;
3. geracao de superpixels SLIC;
4. segmentacao baseada em atributos dos superpixels;
5. morfologia matematica;
6. preenchimento conservador de pequenos contornos;
7. Watershed com marcadores;
8. avaliacao quantitativa por contagem de objetos e proporcao de foreground.

As funcoes centrais de morfologia, SLIC, Otsu, Sobel, componentes conexos e Watershed foram implementadas **from scratch** com NumPy e logica propria.

## 2. Contexto da aplicacao

As imagens contem hemacias com tons rosa-arroxeados, leucocitos azulados, celulas parcialmente sobrepostas, variacao de iluminacao e artefatos de aquisicao. Uma unica limiarizacao por intensidade nao e suficiente.

A pipeline final usa LAB e HSV de forma complementar:

- `LAB.A` destaca variacoes no eixo verde-vermelho e ajuda a preservar hemacias palidas;
- `HSV.S` ajuda a detectar celulas mais visiveis;
- `LAB.B` ajuda a diferenciar hemacias de regioes azuladas;
- superpixels reduzem decisoes isoladas por pixel;
- morfologia remove ruido e rompe pontes estreitas;
- Watershed separa instancias que permaneceram encostadas.

## 3. Ordem final da pipeline e justificativa

| Etapa | Papel na solucao |
|---|---|
| RGB + grayscale + espacos de cor | Preserva a imagem original e cria entradas 2D explicaveis. |
| Comparacao de canais | Permite justificar a escolha de `LAB.A`. |
| Filtro passa-baixa na frequencia | Suaviza variacoes e ruido antes do gradiente. |
| Sobel manual | Gera magnitude de bordas para auxiliar o Watershed. |
| SLIC manual em LAB | Agrupa pixels semelhantes em regioes locais. |
| Regras sobre superpixels | Inclui hemacias visiveis e palidas e exclui regioes azuladas. |
| Abertura + fechamento + area | Remove ruido, rompe pontes estreitas e descarta regioes pequenas. |
| Preenchimento seletivo | Preenche apenas pequenos contornos internos. |
| Watershed com marcadores por erosao | Separa hemacias encostadas e gera rotulos de instancia. |
| Avaliacao | Registra contagem, foreground, componentes, areas e tempo por etapa. |

In [13]:
from pathlib import Path
from pprint import pprint
import inspect
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import list_image_files
from src.edge_enhancement import convolve2d_from_scratch, sobel_edges
from src.morphology import (
    binary_opening,
    fill_small_contours,
    marker_controlled_watershed,
)
from src.pipeline import PIPELINE_STEPS, build_config, run_pipeline, run_pipeline_for_images
from src.superpixels import slic
from src.thresholding import otsu_threshold_from_values
from src.visualization import show_batch_overlays, show_channel_comparison, show_pipeline_evolution

ImportError: cannot import name 'run_pipeline_for_images' from 'src.pipeline' (c:\Users\joaov\Documents\Projetos\digital-image-processing\image-segmentation-pipeline\src\pipeline.py)

## 4. Configuracao final escolhida

Os parametros abaixo foram escolhidos apos comparar canais e testar variacoes de morfologia e Watershed. Eles ficam concentrados em um unico dicionario para facilitar a demonstracao e novos experimentos.

In [ ]:
FINAL_OVERRIDES = {
    'input_strategy': {
        'primary_segmentation_input': 'channel',
        'channel_source': 'LAB',
        'selected_channel': 'A',
    },
    'frequency_filter': {'enabled': True, 'type': 'low_pass', 'radius': 35},
    'edge_enhancement': {
        'enabled': True,
        'method': 'sobel',
        'input': 'frequency_filtered',
        'use_as_watershed_gradient': True,
    },
    'superpixels': {
        'enabled': True,
        'input': 'LAB',
        'num_superpixels': 450,
        'compactness': 10,
        'max_iter': 6,
    },
    'segmentation': {'method': 'rbc_superpixel_rules'},
    'morphology': {
        'enabled': True,
        'opening_size': 5,
        'closing_size': 3,
        'min_object_area': 700,
        'fill_holes': False,
        'connectivity': 8,
    },
    'contour_filling': {
        'enabled': True,
        'max_contour_area_to_fill': 250,
        'connectivity': 8,
    },
    'watershed': {
        'enabled': True,
        'marker_strategy': 'erosion',
        'erosion_iterations': 12,
        'use_edge_gradient': True,
        'ensure_component_coverage': True,
    },
    'output_mask_source': 'watershed_mask',
    'output': {
        'save_report': True,
        'save_artifacts': True,
        'save_pipeline_figure': True,
        'output_dir': 'outputs/pipeline_final',
    },
}

FINAL_CONFIG = build_config(FINAL_OVERRIDES)
print(json.dumps(FINAL_CONFIG, indent=2))

### Regras de classificacao dos superpixels

A decisao final nao usa apenas um limiar. Ela combina grupos de regras explicaveis:

- `visible_rbc`: seleciona hemacias bem visiveis com `mean_s >= 20` e `mean_b >= 115`;
- `pale_rbc`: recupera hemacias palidas com `mean_a >= 129` e `mean_b >= 125`;
- `blue_cells`: exclui regioes mais azuladas com `mean_s >= 15` e `mean_b <= 120`, geralmente associadas a leucocitos.

Cada valor e calculado sobre um superpixel SLIC, nao sobre um pixel isolado.

## 5. Imagens utilizadas

In [ ]:
image_paths = list_image_files(PROJECT_ROOT / 'data' / 'samples' / 'images')
print(f'{len(image_paths)} imagens encontradas:')
for image_path in image_paths:
    print('-', image_path.name)

## 6. Execucao final em lote

A celula seguinte executa **todos** os passos oficiais sobre todas as amostras disponiveis. Como SLIC e morfologia foram implementados manualmente, a execucao pode levar alguns minutos.

In [ ]:
results, batch_summary = run_pipeline_for_images(
    image_paths,
    overrides=FINAL_OVERRIDES,
    steps=PIPELINE_STEPS,
)

print('Passos executados:')
for index, step in enumerate(PIPELINE_STEPS, start=1):
    print(f'{index:02d}. {step}')

In [ ]:
print('Resumo final do lote:')
for row in batch_summary:
    print(
        f"{row['image']}: {row['estimated_rbc_count']} hemacias | "
        f"componentes antes do Watershed={row['components_before_watershed']} | "
        f"foreground={row['final_foreground_ratio']:.3f} | "
        f"tempo={row['total_seconds']:.2f}s"
    )

### Resultado de referencia validado

Na execucao final realizada em **31 de maio de 2026**, a pipeline estimou:

| Imagem | Componentes antes do Watershed | Hemacias estimadas apos Watershed | Foreground final |
|---|---:|---:|---:|
| `BloodImage_00340.jpg` | 18 | 27 | 0.396 |
| `BloodImage_00367.jpg` | 8 | 18 | 0.433 |
| `BloodImage_00368.jpg` | 5 | 27 | 0.510 |

Os valores sao estimativas sem mascara de referencia e devem ser analisados junto dos overlays.

In [ ]:
show_batch_overlays(results)

## 7. Analise detalhada de uma imagem representativa

A primeira imagem e usada como exemplo completo. Os relatorios JSON e as figuras de todas as imagens sao salvos em `outputs/pipeline_final`.

In [ ]:
representative = results[0]
print('Imagem representativa:', Path(representative['image_path']).name)
print('Hemacias estimadas:', representative['estimated_rbc_count'])
print('Relatorio detalhado:', representative['report_path'])

### Comparacao de canais

A mesma implementacao manual de Otsu e aplicada a diferentes canais. A comparacao permite justificar visualmente por que `LAB.A` participa da versao final.

In [ ]:
show_channel_comparison(representative['channel_comparison'])

### Evolucao completa da pipeline

O painel abaixo mostra as entradas, as operacoes realizadas, os mapas auxiliares e a contagem final.

In [ ]:
show_pipeline_evolution(representative)

## 8. Metricas e informacoes coletadas por etapa

O relatorio guarda dimensoes, tipos, estatisticas de intensidade, quantidade de superpixels, regras aplicadas, componentes, areas, foreground, contornos preenchidos, marcadores, instancias e tempo de cada passo.

In [ ]:
pprint(representative['execution_report']['summary'])
print('\nMetricas detalhadas por etapa:')
pprint(representative['stage_metrics'])

## 9. Codigos importantes da implementacao

As celulas abaixo exibem funcoes centrais para facilitar a explicacao tecnica durante a apresentacao.

In [ ]:
print('=== Convolucao manual usada pelo Sobel ===')
print(inspect.getsource(convolve2d_from_scratch))
print('=== SLIC from scratch ===')
print(inspect.getsource(slic))

In [ ]:
print('=== Otsu manual ===')
print(inspect.getsource(otsu_threshold_from_values))
print('=== Abertura morfologica manual ===')
print(inspect.getsource(binary_opening))
print('=== Preenchimento conservador de contornos ===')
print(inspect.getsource(fill_small_contours))
print('=== Watershed manual ===')
print(inspect.getsource(marker_controlled_watershed))

## 10. Arquivos gerados

Para cada imagem, a pipeline salva uma pasta com:

- imagem original, grayscale e canal selecionado;
- filtragem em frequencia e magnitude Sobel;
- superpixels;
- mascara inicial e historico morfologico;
- contornos preenchidos;
- marcadores, mapa de distancia e instancias Watershed;
- mascara binaria final e overlays;
- `config.json` e `execution_report.json`.

Tambem sao salvos `batch_summary.csv` e `batch_summary.json` com a contagem de todas as imagens.

In [ ]:
output_dir = PROJECT_ROOT / FINAL_CONFIG['output']['output_dir']
print('Diretorio de saida:', output_dir)
for path in sorted(output_dir.rglob('*')):
    if path.is_file():
        print('-', path.relative_to(PROJECT_ROOT))

## 11. Discussao critica e limitacoes

A pipeline identifica a maior parte das hemacias visiveis e usa Watershed para separar regioes conectadas. Ainda existem limitacoes importantes:

- celulas cortadas nas bordas podem ser contadas como instancias;
- aglomerados densos continuam dificeis de separar perfeitamente;
- pequenos residuos proximos a leucocitos podem permanecer na mascara;
- sem mascaras de referencia, Dice e IoU nao podem ser calculados de forma honesta;
- a contagem deve ser interpretada como estimativa e acompanhada do overlay visual.

Esses pontos sao relevantes para a discussao dos resultados: a configuracao busca um equilibrio entre recuperar hemacias palidas e evitar a inclusao de regioes azuladas.

## 12. Teste rapido com uma imagem nova

Durante a apresentacao, altere apenas o caminho abaixo. A mesma configuracao final sera aplicada sem recalibracao manual.

In [ ]:
def test_new_image(image_path):
    result = run_pipeline(image_path, overrides=FINAL_OVERRIDES, steps=PIPELINE_STEPS)
    print('Imagem:', Path(image_path).name)
    print('Hemacias estimadas:', result['estimated_rbc_count'])
    print('Relatorio:', result['report_path'])
    show_pipeline_evolution(result)
    return result

# Exemplo:
# new_result = test_new_image(PROJECT_ROOT / 'data' / 'samples' / 'images' / 'BloodImage_00368.jpg')

## 13. Referencias principais

- Achanta et al. - SLIC Superpixels Compared to State-of-the-Art Superpixel Methods. DOI: `10.1109/TPAMI.2012.120`.
- Otsu - A Threshold Selection Method from Gray-Level Histograms. DOI: `10.1109/TSMC.1979.4310076`.
- Meyer - Topographic distance and watershed lines. DOI: `10.1016/0165-1684(94)90060-4`.
- Blood Cell Segmentation Dataset: `https://www.kaggle.com/datasets/paultimothymooney/blood-cells`.